# Load and Combine Each Patient File

In [ ]:
import os
import pandas as pd
from glob import glob
from tqdm import tqdm

# 1. Load all PSV files

from pathlib import Path\n",
    "_repo = Path.cwd().parent if Path.cwd().name == \"preprocessing\" else Path.cwd()\n",
    "data_path = str(_repo / \"dataset_raw\")\n",
psv_files = glob(os.path.join(data_path, "*.psv"))

print(f"Total files found: {len(psv_files)}")

all_patients = []

for file in tqdm(psv_files):
    try:
        df = pd.read_csv(file, sep='|')
        
        # Extract patient ID from filename
        patient_id = os.path.basename(file).replace(".psv", "")
        df["patient_id"] = patient_id
        
        all_patients.append(df)
        
    except Exception as e:
        print(f"Error reading {file}: {e}")

# Combine all patients into one dataframe
full_df = pd.concat(all_patients, ignore_index=True)

print("Combined dataset shape:", full_df.shape)

In [ ]:
# full_df.to_csv('full_patient_data.csv', index=False)
full_df.head(5)

# Stratified Balanced Sampling

In [ ]:
patient_labels = (
    full_df.groupby("patient_id")["SepsisLabel"]
    .max()
    .reset_index()
)

patient_labels.columns = ["patient_id", "patient_sepsis_flag"]

print(patient_labels["patient_sepsis_flag"].value_counts())

In [ ]:
# Separate positive and negative patients
positive_patients = patient_labels[patient_labels["patient_sepsis_flag"] == 1]
negative_patients = patient_labels[patient_labels["patient_sepsis_flag"] == 0]

print("Positive patients:", len(positive_patients))
print("Negative patients:", len(negative_patients))

# Choose equal numbers
n_samples = 1790  # balanced

positive_sample = positive_patients.sample(n=n_samples, random_state=42)
negative_sample = negative_patients.sample(n=n_samples, random_state=42)

balanced_patients = pd.concat([positive_sample, negative_sample])

print("Final patient count:", len(balanced_patients))

In [ ]:
balanced_patient_ids = balanced_patients["patient_id"].tolist()

balanced_df = full_df[full_df["patient_id"].isin(balanced_patient_ids)]

print("Balanced dataset shape:", balanced_df.shape)
# balanced_df.to_csv('balanced_patient_data.csv', index=False)